In [1]:
!git clone https://github.com/UniversalDependencies/UD_English-EWT.git

Cloning into 'UD_English-EWT'...
remote: Enumerating objects: 49423, done.
remote: Counting objects: 100% (957/957), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 49423 (delta 910), reused 871 (delta 846), pack-reused 48466 (from 2)
Receiving objects: 100% (49423/49423), 399.58 MiB | 28.17 MiB/s, done.
Resolving deltas: 100% (44022/44022), done.


In [2]:
!pip install transformers datasets seqeval conllu -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [3]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, TrainingArguments, Trainer
import numpy as np
from seqeval.metrics import precision_score, recall_score, f1_score
from conllu import parse

In [4]:
data = open("UD_English-EWT/en_ewt-ud-train.conllu", "r", encoding="utf-8").read()
sentences = parse(data)

tokens_list = []
tags_list = []

for sentence in sentences[:2000]:  #  limit for speed
    tokens = []
    tags = []
    for token in sentence:
        if isinstance(token["id"], int):  # skip special tokens
            tokens.append(token["form"])
            tags.append(token["upos"])
    tokens_list.append(tokens)
    tags_list.append(tags)

In [5]:
unique_tags = list(set(tag for tags in tags_list for tag in tags))

label2id = {tag: i for i, tag in enumerate(unique_tags)}
id2label = {i: tag for tag, i in label2id.items()}

labels_encoded = [[label2id[tag] for tag in tags] for tags in tags_list]

In [6]:
dataset = Dataset.from_dict({
    "tokens": tokens_list,
    "labels": labels_encoded
})

dataset = dataset.train_test_split(test_size=0.1)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []
    for i, label in enumerate(examples["labels"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels
    return tokenized_inputs

tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/1800 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

In [8]:
model = AutoModelForTokenClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=len(unique_tags),
    id2label=id2label,
    label2id=label2id
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [9]:
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,   # fast training
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no"
)

In [10]:
def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_labels = []
    true_preds = []

    for pred, lab in zip(predictions, labels):
        temp_pred = []
        temp_lab = []
        for p_, l_ in zip(pred, lab):
            if l_ != -100:
                temp_pred.append(id2label[p_])
                temp_lab.append(id2label[l_])
        true_preds.append(temp_pred)
        true_labels.append(temp_lab)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall": recall_score(true_labels, true_preds),
        "f1": f1_score(true_labels, true_preds),
    }

In [13]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [14]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

In [15]:
trainer.train()
trainer.evaluate()

Step,Training Loss
50,1.988650
100,0.683015
150,0.330944
200,0.252886


/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PROPN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: PUNCT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: INTJ seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: DET seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: NOUN seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/usr/local/lib/python3.12/dist-packages/seqeval/metrics/sequence_labeling.py:17

{'eval_loss': 0.2120164930820465,
 'eval_precision': 0.9302391799544419,
 'eval_recall': 0.9312998859749145,
 'eval_f1': 0.9307692307692308,
 'eval_runtime': 16.5414,
 'eval_samples_per_second': 12.091,
 'eval_steps_per_second': 0.786,
 'epoch': 2.0}

In [16]:
def predict(sentence):
    tokens = sentence.split()
    inputs = tokenizer(tokens, return_tensors="pt", is_split_into_words=True)

    outputs = model(**inputs).logits
    predictions = np.argmax(outputs.detach().numpy(), axis=2)

    word_ids = inputs.word_ids()
    final_preds = []

    for i, word_idx in enumerate(word_ids):
        if word_idx is not None:
            final_preds.append(id2label[predictions[0][i]])

    return list(zip(tokens, final_preds))


print(predict("John works at Google in California"))

[('John', 'PROPN'), ('works', 'VERB'), ('at', 'ADP'), ('Google', 'PROPN'), ('in', 'ADP'), ('California', 'PROPN')]


Task 7: Comparison (POS Tagging vs Chunking)
Comparison between POS Tagging and Chunking
  POS Tagging (Part-of-Speech Tagging)
Assigns grammatical labels to each word
Works at word level
Example tags:
NOUN, VERB, ADJ, ADV, PRON

  Chunking (Phrase Detection)
Groups words into meaningful phrases
Works at phrase level
Example tags:
B-NP (Beginning of Noun Phrase)
I-NP (Inside Noun Phrase)
B-VP (Verb Phrase)


Task 8: Report / Blog Content
1. Introduction

In this assignment, a token classification system was developed using transformer models such as DistilBERT to perform Part-of-Speech (POS) tagging and understand sequence labeling tasks. The project demonstrates how modern NLP models handle token-level predictions efficiently.

2. Differences between POS Tagging and Chunking

POS tagging assigns grammatical labels to individual words, while chunking groups words into meaningful phrases. POS tagging is simpler and focuses on syntax, whereas chunking captures higher-level sentence structure.

3. Challenges Faced
* Label Alignment
Handling subword tokens created by tokenizer
Assigning -100 to ignored tokens
* Variable Sequence Length
Sentences had different lengths
Required padding using data collator
* Model Training
Training transformer models is computationally expensive
Reduced dataset size for faster execution

4. Observations
Transformer models provide high accuracy for token classification
Proper preprocessing significantly improves performance
Handling subwords correctly is crucial
Even with limited data, good results can be achieved

5. Conclusion

This project demonstrates the effectiveness of transformer-based models in solving NLP token classification tasks. POS tagging serves as a foundation for more complex tasks like chunking and named entity recognition. Proper handling of tokenization and label alignment is essential for achieving high performance.